In [2]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModel 

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('ok')


ok


In [5]:
from deutschland import klinikatlas
from deutschland.klinikatlas.api import default_api


BASE_URL = "https://bundes-klinik-atlas.de"

configuration = klinikatlas.Configuration(
    host=BASE_URL
)

with klinikatlas.ApiClient(configuration) as api_client:
    api = default_api.DefaultApi(api_client)

    icd_data = api.fileadmin_json_icd_codes_json_get()

In [6]:
tokenizer = AutoTokenizer.from_pretrained("permediq/SapBERT-DE", use_fast=True)
model = AutoModel.from_pretrained("permediq/SapBERT-DE").to(device)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [13]:
bs = 32 # batch size 
all_embs = []
print(type(icd_data[0]['description']))
print(type(icd_data[idx]['description'] for idx in range(32)))

for i in tqdm(np.arange(0, len(icd_data), bs)):
    batch = icd_data[i:i+bs]

    descriptions = [item["description"] for item in batch]


    toks = tokenizer(
        descriptions,
        padding="max_length",
        max_length=40,
        truncation=True,
        return_tensors="pt"
    )
    
    tokse = {}
    for k,v in toks.items():
        tokse[k] = v.to(device)
    cls_rep = model(**tokse)[0][:,0,:] 
    all_embs.append(cls_rep.cpu().detach())
all_embs = torch.cat(all_embs)


<class 'str'>
<class 'generator'>


100%|██████████| 524/524 [01:54<00:00,  4.59it/s]


In [14]:
def cos_sim(a, b):
    a_norm = torch.nn.functional.normalize(a, p=2, dim=1)
    b_norm = torch.nn.functional.normalize(b, p=2, dim=1)
    return torch.mm(a_norm, b_norm.transpose(0, 1))

# cosine similarity of first entity with all the entities



In [15]:
testing = cos_sim(all_embs[0].unsqueeze(0), all_embs)
print(testing)
print(np.argmax(testing))

tensor([[1.0000, 0.9283, 0.7229,  ..., 0.3239, 0.3501, 0.3125]])
tensor(0)


In [17]:
x='heart'
this_embed =[]
toks = tokenizer(x,
                 padding = 'max_length',
                 max_length =40,
                 truncation=True,
                 return_tensors='pt')
tokse = {}
for k,v in toks.items():
    tokse[k] = v.to(device)
cls_rep = model(**tokse)[0][:,0,:] 
this_embed.append(cls_rep.cpu().detach())
this_embed = torch.cat(this_embed)
closest = cos_sim(this_embed, all_embs)
print(closest)
k = np.argmax(closest).item()
print(k)
print(icd_data[k]['description'])
print(icd_data[k]['icdcode'])

tensor([[0.2473, 0.2226, 0.1751,  ..., 0.2537, 0.2719, 0.2421]])
1697
Gutartige Neubildung: Herz
D15.1
